# Visualisation du Blueprint

Ce notebook montre comment visualiser le graphe de dépendances généré par le Blueprint.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from seapopym.blueprint import Blueprint
from seapopym.blueprint.nodes import DataNode, ComputeNode

bp = Blueprint()
bp.register_forcing("temperature")

# 1. Groupe Tuna
# grow_tuna(temperature) -> Tuna_biomass
def grow_tuna(temperature): 
    return temperature

bp.register_group("Tuna", [
    {
        "func": grow_tuna,
        "output_mapping": {"biomass": "biomass"}  # -> Tuna_biomass
    }
])

# 2. Groupe Shark
# Il a besoin de manger du thon.
# Sa fonction attend 'food'. On doit mapper 'food' -> 'Tuna_biomass'.
# Cette fonction retourne plusieurs sorties
def grow_shark(food, temperature): 
    return {"growth": food * (1 / temperature), "biomass": food * temperature}

bp.register_group("Shark", [
    {
        "func": grow_shark,
        "input_mapping": {"food": "Tuna_biomass"},  # Mapping explicite vers l'autre groupe
        "output_mapping": {"growth": "temp_growth", "biomass": "biomass"}  # Sorties multiples
    }
])

plan = bp.build()
print("Plan construit avec succès !")
print("Séquence :", [node.name for node in plan.task_sequence])
print("Variables initiales :", plan.initial_variables)
print("Variables produites :", plan.produced_variables)

In [ ]:
# --- Visualisation ---
plt.figure(figsize=(14, 10))

G = bp.graph
pos = nx.spring_layout(G, seed=42, k=2)

# Séparation des noeuds par type
data_nodes = [n for n in G.nodes if isinstance(n, DataNode)]
compute_nodes = [n for n in G.nodes if isinstance(n, ComputeNode)]

# Distinguer les variables initiales des produites
initial_data_nodes = [n for n in data_nodes if n.name in plan.initial_variables]
produced_data_nodes = [n for n in data_nodes if n.name in plan.produced_variables]

# Dessin des noeuds
nx.draw_networkx_nodes(G, pos, nodelist=initial_data_nodes, 
                       node_color='lightgreen', node_shape='o', 
                       node_size=800, label='Data (Initial)')
nx.draw_networkx_nodes(G, pos, nodelist=produced_data_nodes, 
                       node_color='lightblue', node_shape='o', 
                       node_size=800, label='Data (Produced)')
nx.draw_networkx_nodes(G, pos, nodelist=compute_nodes, 
                       node_color='orange', node_shape='s', 
                       node_size=1000, label='Compute')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, 
                       edge_color='gray', width=2)

# Labels
labels = {n: n.name for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=9, font_weight='bold')

plt.title("Blueprint Dependency Graph\nTuna → Shark (Inter-group dependency)", 
          fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()